# L1 Language Models, the Chat Format and Tokens

## Setup
#### Load the API key and relevant Python libaries.
In this course, we've provided some code that loads the OpenAI API key for you.

In [8]:
import os
from google import genai

# Option 1: Directly set your API key
client = genai.Client(api_key="AQ.Ab8RN6KZio1-ymF4ORFaIv1CyU9Ly_KEMCSTB4IrWDeM4JG0_w")

# Option 2: Read API key from environment variable (recommended)
# client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

#### helper function
Low Temperature (e.g., 0.2): The model tends to produce more focused and deterministic output. It is more likely to choose the most probable next word based on its training data.

Medium Temperature (e.g., 0.5): A balance between randomness and focus. It allows for some variability in the output, making it less predictable than low temperature but not as random as high temperature.

High Temperature (e.g., 0.8 or 1.0): The output becomes more creative and diverse. The model is more likely to introduce less common words and phrases, resulting in more varied and sometimes unpredictable output.

In [9]:
from google import genai

client = genai.Client(api_key="AQ.Ab8RN6KZio1-ymF4ORFaIv1CyU9Ly_KEMCSTB4IrWDeM4JG0_w")

def get_completion(prompt, model="gemini-2.5-flash"):
    response = client.models.generate_content(
        model=model,
        contents=prompt, 
    )
    return response.text

## Prompt the model and get a completion

In [10]:
response = get_completion("What is the capital of France?")

In [11]:
print(response)

The capital of France is **Paris**.


## Tokens

In [12]:
response = get_completion("Take the letters in lollipop \
and reverse them")
print(response)

The letters in "lollipop" reversed are **pollipol**.


"lollipop" in reverse should be "popillol"

In [13]:
response = get_completion("""Take the letters in \
l-o-l-l-i-p-o-p and reverse them""")

In [14]:
response

'popillol'

## Helper function (chat format)
Here's the helper function we'll use in this course.

In [17]:
from google.genai import types

def get_completion_from_messages(messages,
                                 model="gemini-2.5-flash",
                                 temperature=0,
                                 max_tokens=500):

    prompt = "\n".join(
        [f"{m['role']}: {m['content']}" for m in messages]
    )

    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    return response.text



In [18]:
messages =  [  
{'role':'system', 
 'content':"""You are an assistant who\
 responds in the style of Dr Seuss."""},    
{'role':'user', 
 'content':"""write me a very short poem\
 about a happy carrot"""},  
] 
response = get_completion_from_messages(messages, temperature=1)
print(response)

Oh, the carrot named Carl! What a jumpity fellow!
So orange and bright, not


In [19]:
# length
messages =  [  
{'role':'system',
 'content':'All your responses must be \
one sentence long.'},    
{'role':'user',
 'content':'write me a story about a happy carrot'},  
] 
response = get_completion_from_messages(messages, temperature =1)
print(response)

Crisp and vibrant, Carl the carrot beamed as he soaked up the sun, dreaming of becoming the star of a delicious salad.


In [20]:
# combined
messages =  [  
{'role':'system',
 'content':"""You are an assistant who \
responds in the style of Dr Seuss. \
All your responses must be one sentence long."""},    
{'role':'user',
 'content':"""write me a story about a happy carrot"""},
] 
response = get_completion_from_messages(messages, 
                                        temperature =1)
print(response)

Oh, the happiest carrot, named Kip, with a skip and a dip, would dance in the garden, a grin on his lip!


In [21]:
from google.genai import types

def get_completion_and_token_count(messages,
                                   model="gemini-2.5-flash",
                                   temperature=0,
                                   max_tokens=500):

    prompt = "\n".join(
        [f"{m['role']}: {m['content']}" for m in messages]
    )

    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens
        )
    )

    content = response.text

    token_dict = {
        "prompt_tokens": response.usage_metadata.prompt_token_count,
        "completion_tokens": response.usage_metadata.candidates_token_count,
        "total_tokens": response.usage_metadata.total_token_count,
    }

    return content, token_dict

In [22]:
messages = [
{'role':'system', 
 'content':"""You are an assistant who responds\
 in the style of Dr Seuss."""},    
{'role':'user',
 'content':"""write me a very short poem \ 
 about a happy carrot"""},  
] 
response, token_dict = get_completion_and_token_count(messages)

In [23]:
print(response)

Oh, the carrot, so happy and bright,
It wiggled with all of its


In [24]:
print(token_dict)

{'prompt_tokens': 33, 'completion_tokens': 19, 'total_tokens': 529}


#### Notes on using the OpenAI API outside of this classroom

To install the OpenAI Python library:
```
!pip install openai
```

The library needs to be configured with your account's secret key, which is available on the [website](https://platform.openai.com/account/api-keys). 

You can either set it as the `OPENAI_API_KEY` environment variable before using the library:
 ```
 !export OPENAI_API_KEY='sk-...'
 ```

Or, set `openai.api_key` to its value:

```
import openai
openai.api_key = "sk-..."
```

#### A note about the backslash
- In the course, we are using a backslash `\` to make the text fit on the screen without inserting newline '\n' characters.
- GPT-3 isn't really affected whether you insert newline characters or not.  But when working with LLMs in general, you may consider whether newline characters in your prompt may affect the model's performance.